# 06 · Walk-forward validation: which model estimates a NEW season best?
Each season is predicted by models trained **only on earlier seasons**:

| Train | Test |
|---|---|
| 2022A – 2023B | 2024A |
| 2022A – 2024A | 2024B |
| 2022A – 2024B | 2025A |
| 2022A – 2025A | 2025B |

Compared: three no-model baselines (national mean, the district's own history, the district's result last year), Ridge, Random Forest, scikit-learn GBM, LightGBM (± district as a feature), XGBoost/CatBoost if installed, an ensemble, and a **hybrid** (district history × the season anomaly estimated by the GBM).

Scored on **district × crop yields** (survey-weighted, ≥ 20 plots), the level NISR publishes. This is an end-of-season estimate (the season's own inputs and satellite data are used), not a pre-planting forecast. Code: `src/compare_models.py`.

In [ ]:
import os, sys, importlib
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # run from the repository root
sys.path.insert(0, "src"), sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import train_model as tm; tm = importlib.reload(tm)
import compare_models as cm; cm = importlib.reload(cm)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
QUICK = False
print("Python:", sys.executable, "| LightGBM:", tm.HAVE_LGB, "| XGBoost:", cm.xgb is not None, "| CatBoost:", cm.cb is not None)

In [ ]:
overall, by_season, by_crop, cells = cm.run("data/model_data/model_dataset.csv", "data/model_outputs/walk_forward",
                                            first_test="2024A", quick=QUICK)
overall.round(3)

## Error and ranking by model (all four test seasons)

In [ ]:
kind = lambda m: "baseline (no model)" if m in ("national_mean", "district_history", "last_year") else "machine learning"
o = overall.sort_values("MAPE_%", ascending=False)
col = ["C0" if kind(m) == "baseline (no model)" else "C1" for m in o.model]
fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
ax[0].barh(o.model, o["MAPE_%"], color=col); ax[0].set_xlabel("district yield error, MAPE (%) - lower is better")
ax[1].barh(o.model, o.R2_within_crop, color=col); ax[1].set_xlabel("R2 within crop - ranking districts, higher is better")
from matplotlib.patches import Patch
ax[1].legend(handles=[Patch(color="C0", label="baseline (no model)"), Patch(color="C1", label="machine learning")], loc="lower right")
fig.suptitle("Walk-forward validation: estimating district yields of an unseen season"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
show = list(overall.model[:6])
for m in show:
    g = by_season[by_season.model == m]
    ax[0].plot(g.group, g["MAPE_%"], marker="o", label=m)
    ax[1].plot(g.group, g.R2_within_crop, marker="o", label=m)
ax[0].set_ylabel("MAPE (%)"); ax[1].set_ylabel("R2 within crop"); ax[0].set_xlabel("test season"); ax[1].set_xlabel("test season")
ax[1].legend(fontsize=8); fig.suptitle("By test season (6 best models)"); plt.tight_layout(); plt.show()
by_crop.pivot(index="model", columns="group", values="MAPE_%").loc[overall.model].round(1)

## Observed vs estimated district yields, latest test season

In [ ]:
last = cells.test_season.max()
best_ml = next(m for m in overall.model if kind(m) == "machine learning")
c = cells[(cells.test_season == last) & cells.crop.isin(["Maize", "Bush bean", "Irish potato"])]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, m in zip(axes, ["district_history", best_ml]):
    for ax, (crop, g) in zip(row, c.groupby("crop")):
        ax.scatter(g.obs, g[m], s=14, alpha=.7)
        lim = [min(g.obs.min(), g[m].min()), max(g.obs.max(), g[m].max())]; ax.plot(lim, lim, "k--", lw=.8)
        ax.set_title(f"{crop} - {m}  (r={np.corrcoef(g.obs, g[m])[0,1]:.2f})", fontsize=9)
        ax.set_xlabel("observed (kg/ha)"); ax.set_ylabel("estimated")
fig.suptitle(f"Test season {last}: district history vs best ML model"); plt.tight_layout(); plt.show()

## How to read this
- If the **district history / last-year** baselines beat the ML models, the survey's own past values are the best guide to a district's yield level; satellite + input data at district level do not predict season-to-season changes well enough to replace the survey.
- The ML model's value is then the **within-season contrast** (what fertilizer, seed, manure change on comparable plots), used for the yield-gap scenarios in notebook 05, not district yield estimation.
- `hybrid` tests whether the model can add the *season anomaly* on top of the district normal.